# 02 — Feature Engineering, Model Training, Evaluation, and MLflow

This notebook covers assignment sections 2–4: preprocessing, two classification models, stratified cross-validation, evaluation metrics, experiment tracking, and saving a reusable preprocessing-plus-model pipeline.

Run from the project root. Results depend on the dataset version and installed package versions; use the metrics produced by your actual run in the report.


In [ ]:
from pathlib import Path
import sys, json

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from heart_disease.data import download_dataset, prepare_dataset, RAW_PATH, PROCESSED_PATH
if not RAW_PATH.exists():
    download_dataset(RAW_PATH)
prepare_dataset(RAW_PATH, PROCESSED_PATH)
print("Dataset ready:", PROCESSED_PATH)


In [ ]:
import pandas as pd
df = pd.read_csv(PROCESSED_PATH)
X = df.drop(columns=["target"])
y = df["target"].astype(int)
print("Features:", X.shape, "Target counts:", y.value_counts().to_dict())


## Preprocessing strategy

The reusable pipeline imputes missing numeric values with the median, scales numeric features, imputes categorical values with the most frequent value, and one-hot encodes categorical features. The preprocessing is fit only on training folds through the scikit-learn `Pipeline`, helping prevent data leakage.


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from heart_disease.train import build_pipeline, RANDOM_STATE

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "roc_auc": "roc_auc"}


In [ ]:
candidates = {
    "logistic_regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, class_weight="balanced",
        random_state=RANDOM_STATE
    ),
}

rows = []
fitted_models = {}
for name, estimator in candidates.items():
    pipeline = build_pipeline(X_train, estimator)
    cv_results = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, error_score="raise")
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    prob = pipeline.predict_proba(X_test)[:, 1]
    row = {
        "model": name,
        "cv_accuracy_mean": cv_results["test_accuracy"].mean(),
        "cv_precision_mean": cv_results["test_precision"].mean(),
        "cv_recall_mean": cv_results["test_recall"].mean(),
        "cv_roc_auc_mean": cv_results["test_roc_auc"].mean(),
        "test_accuracy": accuracy_score(y_test, pred),
        "test_precision": precision_score(y_test, pred, zero_division=0),
        "test_recall": recall_score(y_test, pred, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, prob),
    }
    rows.append(row)
    fitted_models[name] = pipeline

results_df = pd.DataFrame(rows).sort_values("cv_roc_auc_mean", ascending=False)
display(results_df)


## Model selection

The example selects the model with the highest mean cross-validation ROC-AUC. This is a baseline choice, not a universal rule. For screening-style problems, discuss false negatives and recall, calibration, decision thresholds, class imbalance, and the need for clinical/domain expertise. Do not claim clinical suitability from these metrics.


In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_name]
print("Selected model:", best_name)


In [ ]:
# MLflow experiment tracking: log each model's parameters, CV/holdout metrics,
# and the selected pipeline artifact.
import mlflow
import mlflow.sklearn

mlflow.set_experiment("heart-disease-notebook")
for _, row in results_df.iterrows():
    name = row["model"]
    estimator = candidates[name]
    with mlflow.start_run(run_name=name):
        mlflow.log_param("model", name)
        mlflow.log_param("random_state", RANDOM_STATE)
        mlflow.log_param("cv_folds", 5)
        for metric in row.index:
            if metric != "model":
                mlflow.log_metric(metric, float(row[metric]))
        mlflow.sklearn.log_model(fitted_models[name], artifact_path="pipeline")

model_path = PROJECT_ROOT / "models" / "model.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
import joblib
joblib.dump(best_model, model_path)

metrics_path = PROJECT_ROOT / "reports" / "notebook_model_metrics.json"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps({
    "selected_model": best_name,
    "selection_metric": "mean cross-validation ROC-AUC",
    "results": results_df.to_dict(orient="records")
}, indent=2))
print("Saved model:", model_path)
print("Saved metrics:", metrics_path)


## Further work to document

- Tune model hyperparameters and record the search space.
- Compare the selected metric against the actual assignment/business objective.
- Add confidence intervals or repeated cross-validation if appropriate.
- Inspect false positives and false negatives, and document limitations.
- Open MLflow UI from the project root with `mlflow ui --backend-store-uri ./mlruns`.
